# CellViT JSON -> Patch Visualizations (HPC)

This notebook converts CellViT JSON outputs into patch-level visualization PNGs with 3 panels:
1. original image
2. predicted outlines
3. predicted masks

It is designed to match the style used in your other model notebooks, while keeping output quality unchanged (PNG lossless).


In [7]:
import socket, os, torch 

print("host:", socket.gethostname())
print("cuda:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

host: hpctpa3pc0004
cuda: True
device_count: 1
gpu: NVIDIA A30


## Configuration

Set these paths to your HPC dataset/output locations.


In [14]:
from pathlib import Path

# CellViT output JSON (use cells.json for full contours)
CELLVIT_JSON_PATH = Path('/share/lab_teng/trainee/tusharsingh/cell-seg/inference/cellvit/outputs/SAM/RCC_4/RCC_4/cells.json')

# If True, use dataset.csv tile_x/tile_y (recommended for external tiles)
USE_DATASET_TILE_COORDS = True
DATASET_CSV_PATH = Path('/share/lab_soupir/IHC/ideas/HnE_segmentation/data/outputs/RCC4/preprocessing/output/RCC_4.svs/dataset.csv')
DATASET_IMAGE_ROOT = Path('/share/lab_soupir/IHC/ideas/HnE_segmentation/data/outputs/RCC4/preprocessing/output')

# Dataset coordinate controls (important for GigaPath-style tiling)
# GigaPath stores tile_x/tile_y in level-0 coordinates, but PNGs can come from lower levels.
DATASET_COORD_MODE = 'auto'  # 'auto' | 'xy' | 'swap_xy'
DATASET_TILE_SPAN_MODE = 'auto'  # 'auto' | 'image' | 'step' | 'override'
DATASET_TILE_SPAN_OVERRIDE = None  # level-0 span, e.g. 512 for level-1 256px tiles
DATASET_TILE_SCALE_X = 1.0
DATASET_TILE_SCALE_Y = 1.0
DATASET_TILE_OFFSET_X = 0.0
DATASET_TILE_OFFSET_Y = 0.0

# Fallback folder containing patch images (used when USE_DATASET_TILE_COORDS=False)
PATCH_IMAGE_DIR = Path('/share/lab_soupir/IHC/ideas/HnE_segmentation/data/outputs/RCC4/preprocessing/output/RCC_4.svs')
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}

# Output folder for 3-panel visualization PNG files
OUT_DIR = Path('/share/lab_teng/trainee/tusharsingh/cell-seg/inference/cellvit/outputs/RCC_4/SAM/viz')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# CSV logs
SUMMARY_CSV = OUT_DIR / 'cellvit_visualization_summary.csv'
FAILED_CSV = OUT_DIR / 'failed.csv'
MAPPING_REPORT_CSV = OUT_DIR / 'mapping_report.csv'

# Mapping mode from JSON patch_coordinates -> image files.
# Recommended: 'auto'
# Supported explicit values:
#   filename_row_col, filename_col_row,
#   grid_meta_rowcol, grid_meta_colrow,
#   grid_meta_swapped_rowcol, grid_meta_swapped_colrow,
#   grid_cells_rowcol, grid_cells_colrow,
#   grid_cells_swapped_rowcol, grid_cells_swapped_colrow
PATCH_MATCH_MODE = 'auto'

# Contour coordinate mode for fallback CellViT patch-key mode
#   'auto'         -> pick best mode per patch
#   'local'        -> assume contour already patch-local
#   'shift'        -> contour - patch_offset_xy
#   'scaled_shift' -> contour * scale_to_patch - patch_offset_xy
# In dataset tile mode we force 'tile_global'.
CONTOUR_COORD_MODE = 'auto'

# Optional manual override for scale_to_patch.
# None -> infer from metadata as base_mpp/target_patch_mpp
FORCE_COORD_SCALE_TO_PATCH = None

# Optional manual override for metadata patch size in JSON space.
FORCE_METADATA_PATCH_SIZE = None

# If True, render only images that have at least one mapped cell
ONLY_PATCHES_WITH_CELLS = False

# PNG is lossless; compression changes speed/file-size only
PNG_COMPRESSION = 1  # 0 fastest, 9 smallest

# Multiprocessing
MAX_WORKERS = 16
CHUNKSIZE = 8

# Keep full quality: no resizing applied
ADD_TITLES = True


## Helpers


In [15]:
import os
import re
import json
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
from typing import Optional, Tuple

import numpy as np
import imageio.v3 as iio
import pandas as pd
try:
    import cv2
    HAS_CV2 = True
except Exception:
    cv2 = None
    HAS_CV2 = False
from PIL import Image, ImageDraw
try:
    from natsort import natsorted
except Exception:
    def natsorted(x):
        return sorted(x, key=lambda v: str(v))
from tqdm.auto import tqdm
from matplotlib.colors import hsv_to_rgb
from scipy.ndimage import find_objects


def load_cellvit_cells(json_path: Path):
    obj = json.loads(json_path.read_text())
    metadata = obj.get('wsi_metadata', {})
    type_map = obj.get('type_map', {})
    cells = obj.get('cells', [])
    return metadata, type_map, cells


def list_patch_images(patch_dir: Path, image_exts: set[str]):
    image_paths = [p for p in patch_dir.rglob('*') if p.suffix.lower() in image_exts]
    return natsorted(image_paths)


def parse_patch_key_from_filename(stem: str):
    patterns = [
        r'^(?P<a>-?\d+)_(?P<b>-?\d+)$',
        r'^(?P<a>-?\d+)x[_-](?P<b>-?\d+)y$',
        r'^x(?P<a>-?\d+)_y(?P<b>-?\d+)$',
        r'^(?:r|row)(?P<a>-?\d+)[_-](?:c|col)(?P<b>-?\d+)$',
        r'^(?:c|col)(?P<a>-?\d+)[_-](?:r|row)(?P<b>-?\d+)$',
    ]
    s = stem.lower()
    for pat in patterns:
        m = re.match(pat, s)
        if m:
            return int(m.group('a')), int(m.group('b'))
    return None


def group_cells_by_patch(cells):
    grouped = defaultdict(list)
    patch_offset_xy = {}

    for c in cells:
        coords = c.get('patch_coordinates', None)
        if coords is None or len(coords) != 2:
            continue

        key = (int(coords[0]), int(coords[1]))  # [row, col]
        grouped[key].append(c)

        off = c.get('offset_global', None)  # [x_global, y_global]
        if key not in patch_offset_xy and off is not None and len(off) == 2:
            patch_offset_xy[key] = (float(off[1]), float(off[0]))  # np.flip(offset)

    return grouped, patch_offset_xy


def load_tiles_from_dataset(dataset_csv_path: Path, image_root: Optional[Path] = None):
    df = pd.read_csv(dataset_csv_path)
    required_cols = {'image', 'tile_x', 'tile_y'}
    missing = required_cols.difference(df.columns)
    if missing:
        raise ValueError(f'dataset.csv missing columns: {sorted(missing)}')

    if image_root is None:
        image_root = dataset_csv_path.parent.parent

    if 'tile_id' not in df.columns:
        df['tile_id'] = df['image'].apply(lambda p: Path(str(p)).stem)

    def _resolve_image_path(rel_or_abs):
        p = Path(str(rel_or_abs))
        if p.is_absolute():
            return p
        return image_root / p

    df['image_path'] = df['image'].apply(_resolve_image_path)
    df['tile_x'] = df['tile_x'].astype(float)
    df['tile_y'] = df['tile_y'].astype(float)
    return df


def _infer_positive_step(values: np.ndarray) -> Optional[float]:
    uniq = np.unique(values.astype(np.float64))
    diffs = np.diff(np.sort(uniq))
    diffs = diffs[diffs > 0]
    if diffs.size == 0:
        return None
    return float(np.median(diffs))


def build_cells_by_tile_bbox(
    cells,
    tiles_df: pd.DataFrame,
    tile_span_mode: str = 'auto',
    tile_span_override: Optional[float] = None,
    coord_mode: str = 'xy',
    tile_scale_xy: Tuple[float, float] = (1.0, 1.0),
    tile_offset_xy: Tuple[float, float] = (0.0, 0.0),
):
    # Precompute bbox arrays in global slide coordinates
    bbox_xmin = []
    bbox_ymin = []
    bbox_xmax = []
    bbox_ymax = []
    valid_idx = []
    for i, c in enumerate(cells):
        b = c.get('bbox', None)
        if b is None or len(b) != 2:
            continue
        try:
            ymin, xmin = float(b[0][0]), float(b[0][1])
            ymax, xmax = float(b[1][0]), float(b[1][1])
        except Exception:
            continue
        bbox_xmin.append(xmin)
        bbox_ymin.append(ymin)
        bbox_xmax.append(xmax)
        bbox_ymax.append(ymax)
        valid_idx.append(i)

    bbox_xmin = np.asarray(bbox_xmin, dtype=np.float32)
    bbox_ymin = np.asarray(bbox_ymin, dtype=np.float32)
    bbox_xmax = np.asarray(bbox_xmax, dtype=np.float32)
    bbox_ymax = np.asarray(bbox_ymax, dtype=np.float32)

    work_df = tiles_df.copy()
    if coord_mode not in {'xy', 'swap_xy'}:
        raise ValueError(f'Unsupported coord_mode: {coord_mode}')
    if coord_mode == 'swap_xy':
        tx = work_df['tile_x'].to_numpy(copy=True)
        work_df['tile_x'] = work_df['tile_y'].to_numpy(copy=True)
        work_df['tile_y'] = tx

    sx, sy = float(tile_scale_xy[0]), float(tile_scale_xy[1])
    ox, oy = float(tile_offset_xy[0]), float(tile_offset_xy[1])
    work_df['tile_x'] = work_df['tile_x'].astype(float) * sx + ox
    work_df['tile_y'] = work_df['tile_y'].astype(float) * sy + oy

    step_x = _infer_positive_step(work_df['tile_x'].to_numpy(dtype=np.float64))
    step_y = _infer_positive_step(work_df['tile_y'].to_numpy(dtype=np.float64))

    size_cache = {}

    def get_tile_wh(img_path: Path):
        key = str(img_path)
        if key not in size_cache:
            arr = iio.imread(img_path)
            h, w = arr.shape[:2]
            size_cache[key] = (float(w), float(h))
        return size_cache[key]

    def choose_span(w_img: float, h_img: float) -> Tuple[float, float]:
        if tile_span_override is not None:
            s = float(tile_span_override)
            return s, s
        if tile_span_mode == 'image':
            return float(w_img), float(h_img)
        if tile_span_mode == 'step':
            return float(step_x if step_x is not None else w_img), float(step_y if step_y is not None else h_img)
        if tile_span_mode == 'auto':
            w = float(w_img)
            h = float(h_img)
            if step_x is not None and step_x > 0:
                w = max(w, float(step_x))
            if step_y is not None and step_y > 0:
                h = max(h, float(step_y))
            return w, h
        if tile_span_mode == 'override':
            raise ValueError('tile_span_mode="override" requires tile_span_override to be set')
        raise ValueError(f'Unsupported tile_span_mode: {tile_span_mode}')

    cells_by_tile = {}
    tile_meta = {}

    for _, row in work_df.iterrows():
        tile_id = str(row['tile_id'])
        img_path = Path(row['image_path'])
        x0 = float(row['tile_x'])
        y0 = float(row['tile_y'])
        w_img, h_img = get_tile_wh(img_path)
        w_span, h_span = choose_span(w_img, h_img)
        x1 = x0 + w_span
        y1 = y0 + h_span

        # bbox intersects tile rectangle
        inter = (
            (bbox_xmax > x0)
            & (bbox_xmin < x1)
            & (bbox_ymax > y0)
            & (bbox_ymin < y1)
        )
        idx_local = np.where(inter)[0]
        cells_for_tile = [cells[valid_idx[j]] for j in idx_local]

        cells_by_tile[tile_id] = cells_for_tile
        tile_meta[tile_id] = {
            'image_path': img_path,
            'x0': x0,
            'y0': y0,
            'w': w_span,
            'h': h_span,
            'w_img': w_img,
            'h_img': h_img,
        }

    span_info = {
        'coord_mode': coord_mode,
        'tile_span_mode': tile_span_mode,
        'tile_span_override': float(tile_span_override) if tile_span_override is not None else None,
        'tile_step_x': step_x,
        'tile_step_y': step_y,
        'tile_scale_x': sx,
        'tile_scale_y': sy,
        'tile_offset_x': ox,
        'tile_offset_y': oy,
    }

    return cells_by_tile, tile_meta, span_info


def _derive_grid_map(image_paths, rows: int, cols: int, key_order: str):
    image_map = {}
    if rows <= 0 or cols <= 0:
        return image_map

    for i, p in enumerate(image_paths):
        r = i // cols
        c = i % cols
        if r >= rows:
            break
        if key_order == 'rowcol':
            key = (r, c)
        elif key_order == 'colrow':
            key = (c, r)
        else:
            raise ValueError(f'Unknown key_order: {key_order}')
        image_map[key] = p
    return image_map


def build_candidate_maps(image_paths, metadata, cells_by_patch):
    candidates = {}

    parsed = {}
    for p in image_paths:
        k = parse_patch_key_from_filename(p.stem)
        if k is not None:
            parsed[p] = k

    if parsed:
        m_row_col = {}
        m_col_row = {}
        for p, (a, b) in parsed.items():
            m_row_col[(a, b)] = p
            m_col_row[(b, a)] = p
        candidates['filename_row_col'] = m_row_col
        candidates['filename_col_row'] = m_col_row

    meta_rows = int(metadata.get('orig_n_tiles_rows', 0) or 0)
    meta_cols = int(metadata.get('orig_n_tiles_cols', 0) or 0)

    if len(cells_by_patch) > 0:
        max_row = max(k[0] for k in cells_by_patch.keys())
        max_col = max(k[1] for k in cells_by_patch.keys())
        cell_rows = max_row + 1
        cell_cols = max_col + 1
    else:
        cell_rows, cell_cols = 0, 0

    dim_options = [
        ('meta', meta_rows, meta_cols),
        ('meta_swapped', meta_cols, meta_rows),
        ('cells', cell_rows, cell_cols),
        ('cells_swapped', cell_cols, cell_rows),
    ]

    seen = set()
    for label, rows, cols in dim_options:
        if rows <= 0 or cols <= 0:
            continue
        if (rows, cols) in seen:
            continue
        seen.add((rows, cols))

        candidates[f'grid_{label}_rowcol'] = _derive_grid_map(image_paths, rows, cols, 'rowcol')
        candidates[f'grid_{label}_colrow'] = _derive_grid_map(image_paths, rows, cols, 'colrow')

    return candidates


def score_image_map(name, image_map, cells_by_patch):
    cell_patch_keys = set(cells_by_patch.keys())
    image_keys = set(image_map.keys())
    matched_keys = cell_patch_keys.intersection(image_keys)
    matched_cells = sum(len(cells_by_patch[k]) for k in matched_keys)

    total_cells = sum(len(v) for v in cells_by_patch.values())
    return {
        'mode': name,
        'n_images_mapped': int(len(image_map)),
        'n_cell_patches_total': int(len(cell_patch_keys)),
        'n_cell_patches_matched': int(len(matched_keys)),
        'cell_patch_coverage': float(len(matched_keys) / len(cell_patch_keys)) if cell_patch_keys else 0.0,
        'n_cells_total': int(total_cells),
        'n_cells_matched': int(matched_cells),
        'cell_coverage': float(matched_cells / max(1, total_cells)),
    }


def select_image_map(image_paths, metadata, cells_by_patch, mode='auto'):
    candidates = build_candidate_maps(image_paths, metadata, cells_by_patch)
    if len(candidates) == 0:
        raise ValueError('No valid image mapping candidates found. Check PATCH_IMAGE_DIR and patch filename format.')

    report_rows = [score_image_map(name, m, cells_by_patch) for name, m in candidates.items()]
    report_df = pd.DataFrame(report_rows)

    if mode != 'auto':
        if mode not in candidates:
            raise ValueError(f"PATCH_MATCH_MODE='{mode}' is not available. Available: {sorted(candidates.keys())}")
        chosen_mode = mode
    else:
        report_sorted = report_df.sort_values(
            by=['n_cells_matched', 'n_cell_patches_matched', 'n_images_mapped'],
            ascending=[False, False, False],
        )
        chosen_mode = report_sorted.iloc[0]['mode']

    return candidates[chosen_mode], report_df, chosen_mode


def _transform_point(x: float, y: float, mode: str, x_off: float, y_off: float, scale_to_patch: float):
    if mode == 'local':
        return x, y
    if mode == 'shift':
        return x - x_off, y - y_off
    if mode == 'scaled_shift':
        return x * scale_to_patch - x_off, y * scale_to_patch - y_off
    if mode == 'tile_global':
        return x - x_off, y - y_off
    raise ValueError(f'Unknown contour transform mode: {mode}')


def _get_xy_image_scale(h: int, w: int, metadata_patch_size: Optional[int]):
    if metadata_patch_size is None or int(metadata_patch_size) <= 0:
        return 1.0, 1.0
    m = float(metadata_patch_size)
    return float(w) / m, float(h) / m


def choose_contour_mode(
    cells_for_patch,
    patch_offset_xy: Optional[tuple[float, float]],
    scale_to_patch: float,
    h: int,
    w: int,
    metadata_patch_size: Optional[int],
):
    if not cells_for_patch:
        return 'local'

    x_off, y_off = patch_offset_xy if patch_offset_xy is not None else (0.0, 0.0)
    sx, sy = _get_xy_image_scale(h, w, metadata_patch_size)

    modes = ['local', 'shift', 'scaled_shift']
    in_bounds = {m: 0 for m in modes}
    total = 0

    for c in cells_for_patch[:64]:
        contour = c.get('contour', None)
        if contour is None:
            continue
        for pt in contour[:128]:
            if pt is None or len(pt) != 2:
                continue
            x = float(pt[0])
            y = float(pt[1])
            total += 1

            for m in modes:
                lx, ly = _transform_point(x, y, m, x_off, y_off, scale_to_patch)
                xx = lx * sx
                yy = ly * sy
                if 0.0 <= xx < float(w) and 0.0 <= yy < float(h):
                    in_bounds[m] += 1

    if total == 0:
        return 'local'

    ranked = sorted(
        modes,
        key=lambda m: (in_bounds[m], 2 if m == 'scaled_shift' else 1 if m == 'shift' else 0),
        reverse=True,
    )
    return ranked[0]


def to_rgb_uint8(img):
    arr = np.asarray(img)
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    elif arr.ndim == 3 and arr.shape[-1] == 1:
        arr = np.repeat(arr, 3, axis=-1)
    elif arr.ndim == 3 and arr.shape[-1] >= 3:
        arr = arr[..., :3]
    else:
        raise ValueError(f'Unsupported image shape: {arr.shape}')

    if arr.dtype != np.uint8:
        arr = arr.astype(np.float32)
        if arr.max() <= 1.0:
            arr *= 255.0
        arr = np.clip(arr, 0, 255).astype(np.uint8)

    return arr


def build_instance_mask_from_contours(
    cells_for_patch,
    h,
    w,
    patch_offset_xy=None,
    contour_coord_mode='auto',
    scale_to_patch=1.0,
    metadata_patch_size=None,
):
    mask_img = Image.new('I', (w, h), 0)
    draw = ImageDraw.Draw(mask_img)

    if contour_coord_mode == 'auto':
        mode_used = choose_contour_mode(
            cells_for_patch,
            patch_offset_xy,
            scale_to_patch,
            h,
            w,
            metadata_patch_size,
        )
    elif contour_coord_mode in {'local', 'shift', 'scaled_shift', 'tile_global'}:
        mode_used = contour_coord_mode
    else:
        raise ValueError(f'Unknown contour_coord_mode: {contour_coord_mode}')

    x_off, y_off = patch_offset_xy if patch_offset_xy is not None else (0.0, 0.0)
    sx, sy = _get_xy_image_scale(h, w, metadata_patch_size)

    instance_id = 1
    for c in cells_for_patch:
        contour = c.get('contour', None)
        if contour is None or len(contour) < 3:
            continue

        pts = []
        for pt in contour:
            if pt is None or len(pt) != 2:
                continue

            x = float(pt[0])
            y = float(pt[1])
            lx, ly = _transform_point(x, y, mode_used, x_off, y_off, scale_to_patch)

            xx = lx * sx
            yy = ly * sy
            xx = min(max(xx, 0.0), float(w - 1))
            yy = min(max(yy, 0.0), float(h - 1))
            pts.append((xx, yy))

        if len(pts) < 3:
            continue

        draw.polygon(pts, fill=int(instance_id))
        instance_id += 1

    return np.array(mask_img, dtype=np.int32), mode_used, sx, sy


def masks_to_outlines(masks):
    """
    Compute object outlines from instance mask.

    Uses exact Cellpose/CellSAM contour path when cv2 is available.
    Falls back to boundary extraction when cv2 is unavailable.
    """

    if masks.ndim > 3 or masks.ndim < 2:
        raise ValueError("masks_to_outlines expects 2D or 3D array")

    outlines = np.zeros(masks.shape, dtype=bool)

    if masks.ndim == 3:
        for i in range(masks.shape[0]):
            outlines[i] = masks_to_outlines(masks[i])
        return outlines

    if HAS_CV2:
        slices = find_objects(masks.astype(int))

        for i, slc in enumerate(slices):
            if slc is None:
                continue

            sr, sc = slc
            region = (masks[sr, sc] == (i + 1)).astype(np.uint8)

            contours, _ = cv2.findContours(
                region,
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_NONE,
            )

            if len(contours) == 0:
                continue

            pts = np.concatenate(contours, axis=0).squeeze()
            vc = pts[:, 0] + sc.start
            vr = pts[:, 1] + sr.start
            outlines[vr, vc] = True

        return outlines

    # fallback without cv2: label-boundary detection
    m = masks.astype(np.int32, copy=False)
    dv = m[1:, :] != m[:-1, :]
    fg_v = (m[1:, :] > 0) | (m[:-1, :] > 0)
    ev = dv & fg_v
    outlines[1:, :] |= ev
    outlines[:-1, :] |= ev

    dh = m[:, 1:] != m[:, :-1]
    fg_h = (m[:, 1:] > 0) | (m[:, :-1] > 0)
    eh = dh & fg_h
    outlines[:, 1:] |= eh
    outlines[:, :-1] |= eh

    return outlines


    # bounding boxes per object
    slices = find_objects(masks.astype(int))

    for i, slc in enumerate(slices):

        if slc is None:
            continue

        sr, sc = slc

        region = (masks[sr, sc] == (i + 1)).astype(np.uint8)

        contours, _ = cv2.findContours(
            region,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_NONE
        )

        if len(contours) == 0:
            continue

        pts = np.concatenate(contours, axis=0).squeeze()

        vc = pts[:, 0] + sc.start
        vr = pts[:, 1] + sr.start

        outlines[vr, vc] = True

    return outlines



def mask_overlay(img, masks, colors=None):
    """
    Overlay instance masks on an image using HSV coloring.

    Parameters
    ----------
    img : ndarray
        Image [H,W] or [H,W,C]

    masks : ndarray
        Instance mask [H,W]
        0 = background
        1..N = object IDs

    colors : optional ndarray [N,3]
        RGB colors for objects

    Returns
    -------
    RGB image with colored segmentation overlay
    """

    # convert to grayscale brightness
    if img.ndim > 2:
        img_gray = img.astype(np.float32).mean(axis=-1)
    else:
        img_gray = img.astype(np.float32)

    H, W = img_gray.shape

    HSV = np.zeros((H, W, 3), np.float32)

    # brightness channel
    if img_gray.max() > 1:
        val = img_gray / 255.0
    else:
        val = img_gray

    HSV[:, :, 2] = np.clip(val * 1.5, 0, 1)

    # number of instances
    n_masks = int(masks.max())

    if n_masks == 0:
        return (hsv_to_rgb(HSV) * 255).astype(np.uint8)

    # generate random hues (same logic as Cellpose/CellSAM notebook)
    hues = np.linspace(0, 1, n_masks + 1)[np.random.permutation(n_masks)]

    for n in range(n_masks):

        coords = np.where(masks == n + 1)

        if coords[0].size == 0:
            continue

        if colors is None:
            HSV[coords[0], coords[1], 0] = hues[n]
        else:
            HSV[coords[0], coords[1], 0] = colors[n, 0]

        HSV[coords[0], coords[1], 1] = 1.0

    RGB = (hsv_to_rgb(HSV) * 255).astype(np.uint8)

    return RGB




def draw_outlines(image, outlines):
    """
    Draw red boundaries on image.
    """

    img = image.copy()

    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)

    if img.max() <= 1:
        img = (img * 255).astype(np.uint8)

    y, x = np.nonzero(outlines)

    img[y, x] = [255, 0, 0]

    return img


def save_three_panel_png(img, masks, out_path: Path, png_compression=1):
    """
    Save segmentation panels with Cellpose/CellSAM plotting style.
    """
    outlines = masks_to_outlines(masks)
    overlay = mask_overlay(img, masks)
    outline_img = draw_outlines(img, outlines)

    def _to_rgb_uint8(x):
        arr = np.asarray(x)
        if arr.ndim == 2:
            arr = np.stack([arr] * 3, axis=-1)
        if arr.dtype != np.uint8:
            arr = arr.astype(np.float32)
            if arr.max() <= 1:
                arr = arr * 255.0
            arr = np.clip(arr, 0, 255).astype(np.uint8)
        return arr

    img_rgb = _to_rgb_uint8(img)
    outline_rgb = _to_rgb_uint8(outline_img)
    overlay_rgb = _to_rgb_uint8(overlay)

    panel = np.concatenate([img_rgb, outline_rgb, overlay_rgb], axis=1)

    out_path.parent.mkdir(parents=True, exist_ok=True)

    if HAS_CV2:
        cv2.imwrite(
            str(out_path),
            cv2.cvtColor(panel, cv2.COLOR_RGB2BGR),
            [int(cv2.IMWRITE_PNG_COMPRESSION), int(png_compression)],
        )
    else:
        Image.fromarray(panel).save(
            out_path,
            format='PNG',
            compress_level=int(png_compression),
            optimize=False,
        )


def render_one_task(task):
    (
        key,
        image_path_str,
        cells_for_patch,
        patch_offset_xy,
        scale_to_patch,
        metadata_patch_size,
        out_path_str,
        png_compression,
        contour_coord_mode,
    ) = task

    image_path = Path(image_path_str)
    out_base = Path(out_path_str)

    try:
        img = iio.imread(image_path)
        h, w = img.shape[:2]
        mask, coord_mode_used, sx, sy = build_instance_mask_from_contours(
            cells_for_patch,
            h,
            w,
            patch_offset_xy=patch_offset_xy,
            contour_coord_mode=contour_coord_mode,
            scale_to_patch=float(scale_to_patch),
            metadata_patch_size=metadata_patch_size,
        )
        n_instances = int(mask.max())
        n_cells = int(np.unique(mask).size)  # includes background label 0 (matches other notebooks)
        out_path = out_base.parent / f"{out_base.name}_{n_cells}.png"
        save_three_panel_png(img, mask, out_path, png_compression=png_compression)

        return {
            'patch_key': str(key),
            'image': str(image_path),
            'output': str(out_path),
            'image_w': int(w),
            'image_h': int(h),
            'metadata_patch_size': int(metadata_patch_size) if metadata_patch_size else -1,
            'patch_scale_x': float(sx),
            'patch_scale_y': float(sy),
            'json_cells_for_patch': int(len(cells_for_patch)),
            'n_instances': n_instances,
            'n_cells': n_cells,
            'coord_mode_used': coord_mode_used,
            'coord_scale_to_patch': float(scale_to_patch),
            'status': 'ok',
            'error': '',
        }
    except Exception as e:
        return {
            'patch_key': str(key),
            'image': str(image_path),
            'output': str(out_base),
            'image_w': -1,
            'image_h': -1,
            'metadata_patch_size': int(metadata_patch_size) if metadata_patch_size else -1,
            'patch_scale_x': -1.0,
            'patch_scale_y': -1.0,
            'json_cells_for_patch': int(len(cells_for_patch)),
            'n_instances': -1,
            'n_cells': -1,
            'coord_mode_used': 'error',
            'coord_scale_to_patch': float(scale_to_patch),
            'status': 'error',
            'error': str(e),
        }


## Build tasks and run multiprocessing


In [16]:
metadata, type_map, cells_data = load_cellvit_cells(CELLVIT_JSON_PATH)
print('json:', CELLVIT_JSON_PATH)
print('cells:', len(cells_data))
print('type_map:', type_map)
print('metadata patch_size:', metadata.get('patch_size'))
print('metadata orig_n_tiles_rows/cols:', metadata.get('orig_n_tiles_rows'), metadata.get('orig_n_tiles_cols'))

base_mpp = float(metadata.get('base_mpp', 0.0) or 0.0)
target_patch_mpp = float(metadata.get('target_patch_mpp', 0.0) or 0.0)
if FORCE_COORD_SCALE_TO_PATCH is not None:
    coord_scale_to_patch = float(FORCE_COORD_SCALE_TO_PATCH)
elif base_mpp > 0 and target_patch_mpp > 0:
    coord_scale_to_patch = float(base_mpp / target_patch_mpp)
else:
    coord_scale_to_patch = 1.0

if FORCE_METADATA_PATCH_SIZE is not None:
    metadata_patch_size = int(FORCE_METADATA_PATCH_SIZE)
else:
    metadata_patch_size = int(metadata.get('patch_size', 0) or 0)

print('base_mpp:', base_mpp, '| target_patch_mpp:', target_patch_mpp)
print('coord_scale_to_patch:', coord_scale_to_patch)
print('metadata_patch_size_used:', metadata_patch_size)

use_dataset_mode = bool(USE_DATASET_TILE_COORDS and DATASET_CSV_PATH.exists())
print('use_dataset_tile_mode:', use_dataset_mode)

tasks = []

if use_dataset_mode:
    tiles_df = load_tiles_from_dataset(DATASET_CSV_PATH, DATASET_IMAGE_ROOT)
    print('tiles from dataset.csv:', len(tiles_df))

    coord_modes = [DATASET_COORD_MODE] if DATASET_COORD_MODE in {'xy', 'swap_xy'} else ['xy', 'swap_xy']
    dataset_candidates = []
    best = None

    for coord_mode in coord_modes:
        cells_by_tile_i, tile_meta_i, span_info_i = build_cells_by_tile_bbox(
            cells=cells_data,
            tiles_df=tiles_df,
            tile_span_mode=DATASET_TILE_SPAN_MODE,
            tile_span_override=DATASET_TILE_SPAN_OVERRIDE,
            coord_mode=coord_mode,
            tile_scale_xy=(DATASET_TILE_SCALE_X, DATASET_TILE_SCALE_Y),
            tile_offset_xy=(DATASET_TILE_OFFSET_X, DATASET_TILE_OFFSET_Y),
        )

        unique_mapped_ids = set()
        n_tiles_with_cells = 0
        for tid in tiles_df['tile_id'].astype(str):
            cl = cells_by_tile_i.get(tid, [])
            if len(cl) > 0:
                n_tiles_with_cells += 1
                unique_mapped_ids.update(id(c) for c in cl)

        row = {
            'mode': f'dataset_tile_{coord_mode}',
            'n_tiles': int(len(tiles_df)),
            'n_tiles_with_cells': int(n_tiles_with_cells),
            'n_cells_total': int(len(cells_data)),
            'n_cells_unique_mapped': int(len(unique_mapped_ids)),
            'cell_coverage_unique': float(len(unique_mapped_ids) / len(cells_data)) if len(cells_data) else 0.0,
            'tile_span_mode': span_info_i.get('tile_span_mode'),
            'tile_span_override': span_info_i.get('tile_span_override'),
            'tile_step_x': span_info_i.get('tile_step_x'),
            'tile_step_y': span_info_i.get('tile_step_y'),
            'tile_scale_x': span_info_i.get('tile_scale_x'),
            'tile_scale_y': span_info_i.get('tile_scale_y'),
            'tile_offset_x': span_info_i.get('tile_offset_x'),
            'tile_offset_y': span_info_i.get('tile_offset_y'),
        }
        dataset_candidates.append(row)

        score = (row['n_cells_unique_mapped'], row['n_tiles_with_cells'])
        if best is None or score > best['score']:
            best = {
                'score': score,
                'coord_mode': coord_mode,
                'cells_by_tile': cells_by_tile_i,
                'tile_meta': tile_meta_i,
                'span_info': span_info_i,
            }

    assert best is not None
    print('selected dataset coord mode:', best['coord_mode'])
    print('selected tile span mode:', best['span_info'].get('tile_span_mode'))
    print('selected tile step x/y:', best['span_info'].get('tile_step_x'), best['span_info'].get('tile_step_y'))

    cells_by_tile = best['cells_by_tile']
    tile_meta = best['tile_meta']

    if ONLY_PATCHES_WITH_CELLS:
        tiles_df = tiles_df[tiles_df['tile_id'].astype(str).isin([k for k,v in cells_by_tile.items() if len(v) > 0])].copy()

    for _, row in tiles_df.iterrows():
        tile_id = str(row['tile_id'])
        info = tile_meta[tile_id]
        img_path = Path(info['image_path'])

        if not img_path.exists():
            continue

        out_base = OUT_DIR / f"{img_path.stem}"
        tasks.append(
            (
                tile_id,
                str(img_path),
                cells_by_tile.get(tile_id, []),
                (float(info['x0']), float(info['y0'])),
                1.0,
                None,
                str(out_base),
                PNG_COMPRESSION,
                'tile_global',
            )
        )

    map_report_df = pd.DataFrame(dataset_candidates).sort_values(
        by=['n_cells_unique_mapped', 'n_tiles_with_cells'],
        ascending=False,
    )
    map_report_df.to_csv(MAPPING_REPORT_CSV, index=False)
    print('mapping report:', MAPPING_REPORT_CSV)

else:
    image_paths = list_patch_images(PATCH_IMAGE_DIR, IMAGE_EXTS)
    print('patch images:', len(image_paths), '| dir:', PATCH_IMAGE_DIR)

    cells_by_patch, patch_offset_xy = group_cells_by_patch(cells_data)
    print('unique cell patch keys:', len(cells_by_patch))

    image_map, map_report_df, chosen_mode = select_image_map(
        image_paths=image_paths,
        metadata=metadata,
        cells_by_patch=cells_by_patch,
        mode=PATCH_MATCH_MODE,
    )

    map_report_df = map_report_df.sort_values(
        by=['n_cells_matched', 'n_cell_patches_matched', 'n_images_mapped'],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    print('chosen mapping mode:', chosen_mode)
    print(map_report_df[['mode', 'n_cells_matched', 'cell_coverage', 'n_cell_patches_matched', 'cell_patch_coverage', 'n_images_mapped']].head(10))
    map_report_df.to_csv(MAPPING_REPORT_CSV, index=False)
    print('mapping report:', MAPPING_REPORT_CSV)

    mapped_keys = set(image_map.keys())
    cell_keys = set(cells_by_patch.keys())
    matched_keys = mapped_keys.intersection(cell_keys)
    print('matched patch keys:', len(matched_keys), '/', len(cell_keys))

    if len(matched_keys) == 0:
        raise RuntimeError(
            'No patch key overlap between patch images and cells.json. '
            'Check PATCH_IMAGE_DIR and PATCH_MATCH_MODE. Also inspect mapping_report.csv.'
        )

    items = list(image_map.items())
    if ONLY_PATCHES_WITH_CELLS:
        items = [(k, p) for k, p in items if k in cell_keys]

    for key, img_path in items:
        out_base = OUT_DIR / f"{img_path.stem}"
        tasks.append(
            (
                key,
                str(img_path),
                cells_by_patch.get(key, []),
                patch_offset_xy.get(key, None),
                coord_scale_to_patch,
                metadata_patch_size,
                str(out_base),
                PNG_COMPRESSION,
                CONTOUR_COORD_MODE,
            )
        )

print('tasks:', len(tasks))
workers = max(1, min(int(MAX_WORKERS), os.cpu_count() or 1))
print('workers:', workers)

results = []
if workers <= 1:
    for task in tqdm(tasks, total=len(tasks), desc='CellViT visualization (serial)'):
        results.append(render_one_task(task))
else:
    try:
        with ProcessPoolExecutor(max_workers=workers) as ex:
            for row in tqdm(ex.map(render_one_task, tasks, chunksize=CHUNKSIZE), total=len(tasks), desc='CellViT visualization'):
                results.append(row)
    except Exception as e:
        print(f'multiprocessing failed ({e}); falling back to serial execution')
        for task in tqdm(tasks, total=len(tasks), desc='CellViT visualization (serial fallback)'):
            results.append(render_one_task(task))

results_df = pd.DataFrame(results)
results_df.to_csv(SUMMARY_CSV, index=False)

failed_df = results_df[results_df['status'] != 'ok'][['patch_key', 'image', 'error']].copy()
failed_df.to_csv(FAILED_CSV, index=False)

print('summary:', SUMMARY_CSV)
print('failed:', FAILED_CSV, '| n_failed =', len(failed_df))
print('rendered instances > 0:', int((results_df['n_instances'] > 0).sum()) if 'n_instances' in results_df.columns else int((results_df['n_cells'] > 1).sum()), '/', len(results_df))
results_df.head()


json: /share/lab_teng/trainee/tusharsingh/cell-seg/inference/cellvit/outputs/SAM/RCC_4/RCC_4/cells.json
cells: 114533
type_map: {'1': 'Neoplastic', '2': 'Inflammatory', '3': 'Connective', '4': 'Dead', '5': 'Epithelial'}
metadata patch_size: 1024
metadata orig_n_tiles_rows/cols: 45 83
base_mpp: 0.5 | target_patch_mpp: 0.25
coord_scale_to_patch: 2.0
metadata_patch_size_used: 1024
use_dataset_tile_mode: True
tiles from dataset.csv: 2368
selected dataset coord mode: xy
selected tile span mode: auto
selected tile step x/y: 256.0 256.0
mapping report: /share/lab_teng/trainee/tusharsingh/cell-seg/inference/cellvit/outputs/RCC_4/SAM/viz/mapping_report.csv
tasks: 2368
workers: 16


CellViT visualization:   0%|          | 0/2368 [00:00<?, ?it/s]

summary: /share/lab_teng/trainee/tusharsingh/cell-seg/inference/cellvit/outputs/RCC_4/SAM/viz/cellvit_visualization_summary.csv
failed: /share/lab_teng/trainee/tusharsingh/cell-seg/inference/cellvit/outputs/RCC_4/SAM/viz/failed.csv | n_failed = 9
rendered instances > 0: 2335 / 2368


,patch_key,image,output,image_w,image_h,metadata_patch_size,patch_scale_x,patch_scale_y,json_cells_for_patch,n_instances,n_cells,coord_mode_used,coord_scale_to_patch,status,error
0,RCC_4.svs.27144x_01824y,/share/lab_soupir/IHC/ideas/HnE_segmentation/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,256,256,-1,1.0,1.0,12,12,13,tile_global,1.0,ok,
1,RCC_4.svs.27400x_01824y,/share/lab_soupir/IHC/ideas/HnE_segmentation/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,256,256,-1,1.0,1.0,11,11,12,tile_global,1.0,ok,
2,RCC_4.svs.27656x_01824y,/share/lab_soupir/IHC/ideas/HnE_segmentation/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,256,256,-1,1.0,1.0,19,19,20,tile_global,1.0,ok,
3,RCC_4.svs.27912x_01824y,/share/lab_soupir/IHC/ideas/HnE_segmentation/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,256,256,-1,1.0,1.0,13,13,14,tile_global,1.0,ok,
4,RCC_4.svs.28168x_01824y,/share/lab_soupir/IHC/ideas/HnE_segmentation/d...,/share/lab_teng/trainee/tusharsingh/cell-seg/i...,256,256,-1,1.0,1.0,7,7,8,tile_global,1.0,ok,


## Notes

- PNG output is lossless; `PNG_COMPRESSION` affects only speed and file size.
- For maximum quality retention, no resizing is done.
- For GigaPath-style `dataset.csv`, keep `USE_DATASET_TILE_COORDS=True` and inspect `mapping_report.csv`.
- If tiles were extracted at `level > 0`, use `DATASET_TILE_SPAN_MODE='auto'` (or set `DATASET_TILE_SPAN_OVERRIDE`).
- If most tiles have zero cells, try `DATASET_COORD_MODE='auto'` or force `'swap_xy'`.
- If patch-image mapping looks off in fallback patch-key mode, switch `PATCH_MATCH_MODE` and verify naming.
